# Task3：粗定位＋局部精修，固定五折

复用上轮 **512 ResNet34 每折的原图预测**，在原图裁768×768区域，缩到384×384训练局部热图模型。只新增5个精修模型，每折60轮；不重跑原来的十个粗定位模型，不进行本轮80张全量重训。

输入：比赛图片数据。本修正版已内置你2026-10-25提供的轻量ZIP中所需的512逐折粗预测和元数据；**无需另外上传轻量ZIP或20份粗定位权重**。内置的22个CSV/JSON文件有SHA256校验，仍会与比赛图片哈希、标签和折号核对。精修模型独立使用官方ImageNet ResNet34初始化，定位头随机初始化，未从粗模型权重接着训练。

公开参考：原全量单模型12.31624、上轮320＋512等权五折集成12.37299，均为公开榜得分；当前公开最佳仍为12.31624。本版尚无真实训练或公开分数，不保证改进。

研究参考：[ZIAN（原图多尺度ROI的粗到细定位）](https://arxiv.org/abs/2210.02445)。这里是较简单的独立局部热图实现，没有复现其注意力融合架构或使用其外部眼底训练权重。


## 1. 挂载与配置

在Kaggle导入此Notebook，只需添加2026task3比赛图片数据。保持 `COARSE_SOURCE=None`，下一单元会自动还原内置的上轮512粗预测缓存。若明确要用自己另行挂载的原结果，可把 `COARSE_SOURCE` 改为直接含原protocol.json的目录或原ZIP的完整路径。

开启GPU和Internet（首次下载约87 MB官方ImageNet权重）。仅使用cuda:0，五折顺序执行。只保存在新目录，不修改原粗定位结果。当前有本地CPU实验，本Notebook在导入Torch前阻止本地执行。


In [ ]:
from pathlib import Path
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('请在Kaggle GPU环境执行。本机不启动训练，也不导入Torch。')

DATA_DIR = None
COARSE_SOURCE = None  # 默认使用内置的上轮512粗预测；无需添加额外Input
OUTPUT_DIR = Path('/kaggle/working/task3_refine_cv')
RESUME_DIR = None     # 可填本版完整Output目录，里面直接包含本版protocol.json
WEIGHTS = None        # 可填官方resnet34-b627a593.pth，否则自动下载/查缓存
EPOCHS = 60
BATCH_SIZE = 8
CHECKPOINT_EVERY = 5
CPU_THREADS = 2
SELECTION_METRIC = 'rmse_coordinate_px'  # 官方公式仍待确认；另存MRE最优权重
RUN_FOLDS = [0,1,2,3,4]
KEEP_LAST_CHECKPOINT = False
CREATE_FULL_BACKUP = True  # 完成后自动打包含模型的完整ZIP，便于下载


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import csv, gc, hashlib, io, json, math, os, random, shutil, statistics, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
WEIGHT_NAME = 'resnet34-b627a593.pth'
WEIGHT_URL = 'https://download.pytorch.org/models/' + WEIGHT_NAME

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


def contact_sheet(data, records, path, predictions=None):
    rows = records.reset_index(drop=True)
    n = min(len(rows), 20)
    canvas = Image.new('RGB', (1120, math.ceil(n / 4) * 232), '#f3f5f7')
    draw = ImageDraw.Draw(canvas)
    font_path = Path('C:/Windows/Fonts/arial.ttf')
    font = ImageFont.truetype(str(font_path), 16) if font_path.exists() else ImageFont.load_default()
    for i, row in rows.iloc[:n].iterrows():
        with Image.open(data / row['subset'] / row.filename) as im:
            im = im.convert('RGB')
            im.thumbnail((276, 195))
            ox, oy = (i % 4) * 280 + (280 - im.width) // 2, (i // 4) * 232
            canvas.paste(im, (ox, oy))
            points = []
            if 'nx' in row and pd.notna(row.nx):
                points.append((row.nx, row.ny, '#00ffff'))
            if predictions is not None:
                points.append((*predictions[i], '#ffff00'))
            for x, y, color in points:
                cx, cy = ox + x * im.width, oy + y * im.height
                draw.ellipse((cx - 4, cy - 4, cx + 4, cy + 4), outline=color, width=2)
                draw.line((cx - 8, cy, cx + 8, cy), fill=color)
                draw.line((cx, cy - 8, cx, cy + 8), fill=color)
        draw.text(((i % 4) * 280 + 7, (i // 4) * 232 + 199),
                  f'{int(row.image_id):04} | {int(row.width)} x {int(row.height)}', font=font, fill='#15202b')
    canvas.save(path)


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def quantile_linear(values, probability):
    values = sorted(float(v) for v in values)
    if not values:
        raise ValueError('No values.')
    position = (len(values) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return values[low] + (values[high] - values[low]) * (position - low)


def metrics_from_rows(rows):
    rows = list(rows)
    if not rows:
        raise ValueError('No labeled predictions.')
    distance, dx, dy = [], [], []
    for row in rows:
        ex = float(row['Fovea_X']) - float(row['gt_x'])
        ey = float(row['Fovea_Y']) - float(row['gt_y'])
        if not math.isfinite(ex) or not math.isfinite(ey):
            raise ValueError('Non-finite error.')
        dx.append(ex)
        dy.append(ey)
        distance.append(math.hypot(ex, ey))
    return dict(n=len(rows), mean_distance_px=statistics.mean(distance),
                median_distance_px=statistics.median(distance),
                p90_distance_px=quantile_linear(distance, .9), max_distance_px=max(distance),
                mae_x_px=statistics.mean(abs(v) for v in dx), mae_y_px=statistics.mean(abs(v) for v in dy),
                rmse_coordinate_px=math.sqrt(sum(x*x+y*y for x, y in zip(dx, dy)) / (2 * len(rows))))


def validate_oof(rows, records):
    expected = {int(r['image_id']): r for r in records}
    if len(expected) != len(records) or len(rows) != len(records):
        raise ValueError('OOF must contain exactly one row per training image.')
    seen, groups = set(), {}
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen:
            raise ValueError('Unknown or repeated OOF image ID.')
        seen.add(key)
        source = expected[key]
        if int(row['fold']) != int(source['fold']):
            raise ValueError('OOF fold does not match the fixed split.')
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('OOF dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('OOF image hash differs.')
        if abs(float(row['gt_x']) - float(source['x'])) > 1e-6 or abs(float(row['gt_y']) - float(source['y'])) > 1e-6:
            raise ValueError('OOF ground truth differs.')
        groups.setdefault(str(source['pixel_sha256']), set()).add(int(row['fold']))
    if seen != set(expected) or any(len(value) != 1 for value in groups.values()):
        raise ValueError('Incomplete OOF coverage or duplicate group leakage.')
    return sorted(rows, key=lambda r: int(r['image_id']))


def validate_test_rows(rows, records, fold):
    expected = {int(r['image_id']): r for r in records}
    if len(rows) != len(expected):
        raise ValueError('Incomplete test prediction coverage.')
    seen = set()
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen or int(row['fold']) != fold:
            raise ValueError('Invalid test ID/fold.')
        seen.add(key)
        source = expected[key]
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('Test image dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('Test image hash differs.')
        x, y = float(row['Fovea_X']), float(row['Fovea_Y'])
        if not all(math.isfinite(v) for v in [x, y]) or not (0 <= x < int(source['width']) and 0 <= y < int(source['height'])):
            raise ValueError('Test prediction outside image.')
    return {int(r['image_id']): r for r in rows}


def average_test_folds(folds, records, fold_count=5):
    if set(folds) != set(range(fold_count)):
        raise ValueError('All five folds are required before exporting a five-fold submission.')
    mappings = [validate_test_rows(folds[f], records, f) for f in range(fold_count)]
    averaged = []
    for source in records:
        key = int(source['image_id'])
        averaged.append(dict(image_id=key, width=int(source['width']), height=int(source['height']),
                             Fovea_X=statistics.mean(float(m[key]['Fovea_X']) for m in mappings),
                             Fovea_Y=statistics.mean(float(m[key]['Fovea_Y']) for m in mappings)))
    return averaged


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def resolve_weights(output, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates += [Path(torch.hub.get_dir()) / 'checkpoints' / WEIGHT_NAME]
    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent.parent / 'task2/.torch/hub/checkpoints' / WEIGHT_NAME)
    if Path('/kaggle/input').exists():
        candidates += sorted(Path('/kaggle/input').rglob(WEIGHT_NAME))
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        path = output / '_pretrained' / WEIGHT_NAME
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file():
            print('Downloading official ResNet34 ImageNet weights (about 87 MB).', flush=True)
            torch.hub.download_url_to_file(WEIGHT_URL, str(path), hash_prefix='b627a593', progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith('b627a593'):
        raise ValueError(f'ImageNet weight hash mismatch: {path}')
    save_json(output / 'pretrained_source.json', dict(path=str(path), url=WEIGHT_URL, sha256=digest))
    return path


class ResNet34CV(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        if weights is not None:
            self.backbone.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
        c1 = b.layer1(x)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


# Embedded coarse predictions: the original completed 512 CV experiment.
# This only materializes cached CSV/JSON; it neither infers nor trains a model.
import base64
EMBEDDED_COARSE_SHA256 = '0fc2ee733b1ec10d91d2fe1800c0224c2b098f408eaaf4b543a16722135b4fd6'
EMBEDDED_COARSE_B64 = '...'
if COARSE_SOURCE is None:
    _coarse_payload = base64.b64decode(EMBEDDED_COARSE_B64,validate=True)
    if hashlib.sha256(_coarse_payload).hexdigest() != EMBEDDED_COARSE_SHA256:
        raise RuntimeError('内置粗预测缓存校验失败，请重新导入完整Notebook。')
    _coarse_cache = Path('/kaggle/working/task3_coarse512_predictions_cache.zip')
    if not _coarse_cache.exists() or hashlib.sha256(_coarse_cache.read_bytes()).hexdigest() != EMBEDDED_COARSE_SHA256:
        _coarse_cache.write_bytes(_coarse_payload)
    COARSE_SOURCE = _coarse_cache
    print('已载入内置的上轮512五折粗预测，无需额外上传结果ZIP。')
    del _coarse_payload


In [ ]:
from __future__ import annotations
REFINE_SOURCE_SHA256 = {'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6', 'task3_kaggle_refine.py': 'feed93ec81a810165a968715b55155f381f987f13920777819a688e55e8822cb'}
"""Kaggle-only five-fold local refinement using cached 512 coarse predictions."""

import csv
import gc
import hashlib
import io
import json
import math
import os
from pathlib import Path, PurePosixPath
import random
import shutil
import statistics
import sys
import time
import zipfile


import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


REFINE_VERSION = 'resnet34-refine-v1'
REFINE_ARMS = ('coarse512', 'refine384', 'coarse_refine_equal')


class CoarseBundle:
    """Read only metadata/predictions from a directory or ZIP; never extract/load weights."""
    def __init__(self, path):
        self.path, self.prefix, self.archive = Path(path).resolve(), '', None
        if self.path.is_dir():
            return
        if not self.path.is_file() or self.path.suffix.lower() != '.zip':
            raise ValueError(f'Coarse source is not a directory or ZIP: {path}')
        self.archive = zipfile.ZipFile(self.path)
        try:
            candidates = []
            for entry in self.archive.infolist():
                pure = PurePosixPath(entry.filename)
                if pure.is_absolute() or '..' in pure.parts or '\\' in entry.filename:
                    raise ValueError('Unsafe archive member path.')
                if pure.name == 'protocol.json' and entry.file_size < 1000000:
                    value = json.loads(self.archive.read(entry).decode('utf-8-sig'))
                    if value.get('version') == 'resnet34-cv-v1':
                        candidates.append(entry.filename[:-len('protocol.json')])
            if len(candidates) != 1:
                raise ValueError('ZIP must contain exactly one original CV protocol.')
            self.prefix = candidates[0]
        except BaseException:
            self.close()
            raise

    def text(self, relative):
        if self.archive is None:
            path = (self.path / relative).resolve()
            if not path.is_relative_to(self.path):
                raise ValueError('Coarse file escapes its source folder.')
            if path.stat().st_size > 5000000:
                raise ValueError('Unexpectedly large metadata file.')
            return path.read_text(encoding='utf-8-sig')
        name = self.prefix + relative
        if self.archive.getinfo(name).file_size > 5000000:
            raise ValueError('Unexpectedly large metadata member.')
        return self.archive.read(name).decode('utf-8-sig')

    def rows(self, name):
        return list(csv.DictReader(io.StringIO(self.text(name), newline='')))

    def json(self, name):
        return json.loads(self.text(name))

    def close(self):
        if self.archive is not None:
            self.archive.close()
            self.archive = None


def find_coarse_source(explicit=None, working='/kaggle/working', input_root='/kaggle/input'):
    if explicit is not None:
        return Path(explicit).resolve()
    current = Path(working) / 'task3_resnet34_cv'
    if (current/'protocol.json').is_file():
        return current.resolve()
    candidates = []
    root = Path(input_root)
    if root.exists():
        for path in root.rglob('protocol.json'):
            try:
                if json.loads(path.read_text(encoding='utf-8-sig')).get('version') == 'resnet34-cv-v1':
                    candidates.append(path.parent.resolve())
            except (OSError, ValueError):
                continue
        for name in ('task3_cv_results_light.zip', 'task3_cv_full_backup.zip'):
            candidates += [p.resolve() for p in root.rglob(name)]
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f'Attach the previous light results ZIP as an Input, or set COARSE_SOURCE. Found: {candidates}')
    return candidates[0]


def validate_coarse_bundle(source, train_records, test_records):
    bundle = CoarseBundle(source)
    try:
        protocol = bundle.json('protocol.json')
        if protocol.get('version') != 'resnet34-cv-v1' or protocol.get('folds') != 5:
            raise ValueError('Expected the previous five-fold CV experiment.')
        if protocol.get('selection_metric') != 'rmse_coordinate_px':
            raise ValueError('This predeclared experiment reuses the RMSE-selected 512 coarse checkpoints.')
        old_split = bundle.rows('split.csv')
        if len(old_split) != len(train_records):
            raise ValueError('Previous split is incomplete.')
        prior = {int(r['image_id']): r for r in old_split}
        if len(prior) != len(train_records):
            raise ValueError('Duplicate IDs in previous split.')
        for record in train_records:
            old = prior[int(record['image_id'])]
            for field in ('fold', 'width', 'height'):
                if int(old[field]) != int(record[field]):
                    raise ValueError('Coarse split or dimensions differ.')
            if old['pixel_sha256'] != record['pixel_sha256']:
                raise ValueError('Coarse image hash differs.')
            if any(abs(float(old[k])-float(record[k])) > 1e-6 for k in ('x', 'y')):
                raise ValueError('Coarse labels differ.')
        validation, testing, metadata, coverage = {}, {}, [], []
        for fold in range(5):
            prefix = f'resnet34_512/fold_{fold}/'
            config = bundle.json(prefix+'train_config.json')
            done = bundle.json(prefix+'fold_complete.json')
            signature = stable_hash(config)
            if (int(config['fold']) != fold or int(config['image_size']) != 512 or
                    config['data_fingerprint'] != protocol['data_fingerprint'] or
                    config['source_sha256'] != protocol['source_sha256'] or
                    done.get('signature') != signature or not done.get('predictions_complete') or
                    done['training']['signature'] != signature):
                raise ValueError('Coarse fold metadata is inconsistent.')
            held = [r for r in train_records if int(r['fold']) == fold]
            rows = validate_oof(bundle.rows(prefix+'validation_raw.csv'), held)
            # Verify finite in-bounds coordinates as well as OOF membership/labels.
            validate_test_rows(rows, held, fold)
            best = done['training']['best']['rmse_coordinate_px']
            if not math.isclose(metrics_from_rows(rows)['rmse_coordinate_px'], float(best['score']), abs_tol=1e-6):
                raise ValueError('Coarse predictions do not match the selected checkpoint metrics.')
            validation[fold] = {int(r['image_id']): r for r in rows}
            testing[fold] = validate_test_rows(bundle.rows(prefix+'test_raw.csv'), test_records, fold)
            metadata.append(dict(fold=fold, config=config, selected=best))
            coverage.extend(rows)
        fingerprint = stable_hash(dict(protocol=protocol, split=old_split, folds=metadata,
                                        validation=validation, test=testing))
        return dict(validation=validation, test=testing, fingerprint=fingerprint,
                    protocol=protocol, metadata=metadata,
                    baseline_metrics=metrics_from_rows(coverage))
    finally:
        bundle.close()


def crop_geometry(center, side):
    if side < 1 or int(side) != side or not all(math.isfinite(float(v)) for v in center):
        raise ValueError('Invalid crop geometry.')
    left, top = [math.floor(float(v)-side/2) for v in center]
    return (left, top, left+int(side), top+int(side))


def point_to_patch(point, box):
    return ((float(point[0])-box[0])/(box[2]-box[0]),
            (float(point[1])-box[1])/(box[3]-box[1]))


def patch_to_original(point, box):
    return (box[0]+float(point[0])*(box[2]-box[0]),
            box[1]+float(point[1])*(box[3]-box[1]))


def make_refinement_row(record, point, coarse, box, fold, max_shift):
    """Map patch coordinates back before scoring; a fixed fallback needs no labels."""
    width, height = int(record['width']), int(record['height'])
    cx, cy = float(coarse['Fovea_X']), float(coarse['Fovea_Y'])
    if not (0 <= cx < width and 0 <= cy < height):
        raise ValueError('Coarse coordinate outside original image.')
    raw_x, raw_y = patch_to_original(point, box)
    shift = math.hypot(raw_x-cx, raw_y-cy)
    reason = ''
    if not all(math.isfinite(v) for v in (raw_x, raw_y)):
        raise ValueError('Non-finite refinement coordinate.')
    if not (0 <= raw_x < width and 0 <= raw_y < height):
        reason = 'outside_image'
    elif shift > max_shift:
        reason = 'shift_gt_fixed_limit'
    x, y = (cx, cy) if reason else (raw_x, raw_y)
    row = dict(image_id=int(record['image_id']), fold=int(fold), width=width, height=height,
               pixel_sha256=str(record['pixel_sha256']), nx=x/width, ny=y/height,
               Fovea_X=x, Fovea_Y=y, coarse_x=cx, coarse_y=cy,
               raw_refined_x=raw_x, raw_refined_y=raw_y, shift_px=shift,
               fallback=int(bool(reason)), fallback_reason=reason,
               crop_left=box[0], crop_top=box[1], crop_side=box[2]-box[0])
    if record['subset'] == 'train':
        dx, dy = x-float(record['x']), y-float(record['y'])
        row.update(gt_x=float(record['x']), gt_y=float(record['y']), dx=dx, dy=dy,
                   distance_px=math.hypot(dx, dy), squared_error=dx*dx+dy*dy)
    return row


def blend_prediction_rows(coarse_rows, refine_rows, labeled):
    coarse = {int(r['image_id']): r for r in coarse_rows}
    refined = {int(r['image_id']): r for r in refine_rows}
    if len(coarse) != len(coarse_rows) or len(refined) != len(refine_rows) or set(coarse) != set(refined):
        raise ValueError('Blend IDs differ or repeat.')
    result = []
    for key in sorted(coarse):
        a, b = coarse[key], refined[key]
        for field in ('fold', 'width', 'height'):
            if int(a[field]) != int(b[field]):
                raise ValueError('Blend fold or dimensions differ.')
        if a['pixel_sha256'] != b['pixel_sha256']:
            raise ValueError('Blend pixels differ.')
        row = {name:a[name] for name in ('image_id', 'fold', 'width', 'height', 'pixel_sha256')}
        x, y = [(float(a[f'Fovea_{axis}'])+float(b[f'Fovea_{axis}']))/2 for axis in ('X','Y')]
        row.update(Fovea_X=x, Fovea_Y=y, nx=x/int(a['width']), ny=y/int(a['height']))
        if labeled:
            if any(abs(float(a[k])-float(b[k])) > 1e-6 for k in ('gt_x','gt_y')):
                raise ValueError('Blend ground truth differs.')
            dx, dy = x-float(a['gt_x']), y-float(a['gt_y'])
            row.update(gt_x=float(a['gt_x']), gt_y=float(a['gt_y']), dx=dx, dy=dy,
                       distance_px=math.hypot(dx,dy), squared_error=dx*dx+dy*dy)
        result.append(row)
    return result


class OriginalImageCache:
    def __init__(self, data):
        self.data, self.images = Path(data), {}

    def get(self, row):
        key = int(row['image_id'])
        if key not in self.images:
            with Image.open(self.data/row['subset']/row['filename']) as image:
                self.images[key] = image.convert('RGB')
        image = self.images[key]
        if image.size != (int(row['width']), int(row['height'])):
            raise ValueError('Original image dimensions changed.')
        return image


class RefinementDataset(Dataset):
    def __init__(self, records, cache, config, fold, mode, coarse=None):
        self.records, self.cache, self.config = list(records), cache, config
        self.mode, self.coarse = mode, coarse
        if mode not in ('train','validation','test'):
            raise ValueError('Invalid dataset mode.')
        for r in self.records:
            if mode == 'train' and (r['subset'] != 'train' or int(r['fold']) == fold):
                raise ValueError('Held-out labels must not enter refinement training.')
            if mode == 'validation' and (r['subset'] != 'train' or int(r['fold']) != fold):
                raise ValueError('Validation fold differs.')
            if mode == 'test' and r['subset'] != 'test':
                raise ValueError('Test dataset contains labels.')
        if mode != 'train' and (coarse is None or {int(r['image_id']) for r in self.records} != set(coarse)):
            raise ValueError('Inference requires matching cached coarse predictions.')
        self.tensor = transforms.Compose([transforms.ToTensor(),
            transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
        self.jitter = transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.08, hue=.01)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row, cfg = self.records[index], self.config
        if self.mode == 'train':
            center = (float(row['x'])+random.uniform(-cfg['center_jitter'],cfg['center_jitter']),
                      float(row['y'])+random.uniform(-cfg['center_jitter'],cfg['center_jitter']))
        else:
            # No label fields are consulted for validation/test crop geometry.
            prior = self.coarse[int(row['image_id'])]
            center = (float(prior['Fovea_X']), float(prior['Fovea_Y']))
        box = crop_geometry(center, cfg['crop_side'])
        image = self.cache.get(row).crop(box).resize((cfg['image_size'],cfg['image_size']), Image.Resampling.BILINEAR)
        point = np.array(point_to_patch((row['x'],row['y']),box) if self.mode == 'train' else (.5,.5), dtype=np.float32)
        if self.mode == 'train':
            if random.random() < .5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                point[0] = 1-point[0]
            angle, scale = math.radians(random.uniform(-8,8)), random.uniform(.95,1.05)
            a, b = scale*math.cos(angle), -scale*math.sin(angle)
            size, middle = cfg['image_size'], cfg['image_size']/2
            tx, ty = random.uniform(-.02,.02)*size, random.uniform(-.02,.02)*size
            matrix = np.array([[a,b,middle+tx-a*middle-b*middle],
                               [-b,a,middle+ty+b*middle-a*middle],[0,0,1]])
            point = ((matrix @ [point[0]*size,point[1]*size,1])[:2]/size).astype(np.float32)
            if not np.isfinite(point).all() or not ((point > 0).all() and (point < 1).all()):
                raise ValueError('Augmentation moved the label out of the patch.')
            image = image.transform((size,size),Image.Transform.AFFINE,np.linalg.inv(matrix)[:2].reshape(-1),
                                    resample=Image.Resampling.BILINEAR,fillcolor=(0,0,0))
            image = self.jitter(image)
        return self.tensor(image), torch.from_numpy(point), torch.tensor(box,dtype=torch.int64)


@torch.inference_mode()
def predict_refinement(model, loader, device, config, coarse, fold):
    model.eval()
    result, offset = [], 0
    for images, _, boxes in loader:
        with torch.autocast(device_type='cuda',enabled=True):
            logits = model(images.to(device))
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite inference heatmap.')
        points = decode_heatmap(logits, radius=config['decode_radius']).cpu().tolist()
        for point, box in zip(points,boxes.tolist()):
            row = loader.dataset.records[offset]
            result.append(make_refinement_row(row,point,coarse[int(row['image_id'])],box,fold,config['max_shift']))
            offset += 1
    if offset != len(loader.dataset):
        raise ValueError('Refinement inference row count differs.')
    return result


def refinement_sources():
    if 'REFINE_SOURCE_SHA256' in globals():
        return dict(REFINE_SOURCE_SHA256)
    base = Path(__file__).resolve().parent
    return {name:hashlib.sha256((base/name).read_bytes()).hexdigest()
            for name in ('task3_pipeline.py','task3_kaggle_cv.py','task3_kaggle_refine.py')}


def load_refine_checkpoint(path, signature, optimizer=False):
    item = torch.load(path,map_location='cpu',weights_only=not optimizer)
    if item.get('signature') != signature:
        raise ValueError('Refinement checkpoint protocol differs.')
    return item


def validate_refine_config(config):
    for key in ('epochs','batch_size','checkpoint_every','cpu_threads','image_size','crop_side'):
        if int(config[key]) != config[key] or config[key] <= 0:
            raise ValueError(f'{key} must be a positive integer.')
    if config['image_size'] != 384 or config['crop_side'] != 768:
        raise ValueError('This experiment predeclares a 768-pixel native crop resized to 384.')
    if not 0 < config['center_jitter'] <= config['crop_side']/4 or not 0 < config['max_shift'] <= config['crop_side']/2:
        raise ValueError('Invalid crop jitter/fallback limit.')
    if config['selection_metric'] not in ('rmse_coordinate_px','mean_distance_px'):
        raise ValueError('Unsupported metric.')


def refinement_eta(output, epoch, times, config):
    if len(times) < 3:
        return
    pending = sum(not (output/f'fold_{fold}/fold_complete.json').exists()
                  for fold in range(5) if fold != config['fold'])
    rate = statistics.median(times[-8:])
    remaining = (config['epochs']-epoch+pending*config['epochs'])*rate/60
    save_json(output/'eta.json',dict(fold=config['fold'],epoch=epoch,seconds_per_epoch=rate,
        approximate_training_minutes_remaining=remaining,
        excludes='Image loading, future cold starts, checkpoint restore and final reporting; partial folds estimated as full.'))
    if epoch == 3 or epoch % 10 == 0:
        print(f'ETA about {remaining:.1f} training minutes for all five folds.',flush=True)


def train_refine_fold(development, held, cache, coarse, folder, config, weights, output):
    device, fold, selected = torch.device('cuda:0'), config['fold'], config['selection_metric']
    folder.mkdir(parents=True,exist_ok=True)
    signature = stable_hash(config)
    config_file = folder/'train_config.json'
    if config_file.exists() and json.loads(config_file.read_text()) != config:
        raise ValueError('Refinement fold configuration differs.')
    save_json(config_file,config)
    valid_set = RefinementDataset(held,cache,config,fold,'validation',coarse['validation'][fold])
    valid_loader = DataLoader(valid_set,batch_size=config['batch_size'],shuffle=False,num_workers=0)
    complete_file = folder/'training_complete.json'
    if complete_file.exists():
        done = json.loads(complete_file.read_text())
        if done['signature'] != signature:
            raise ValueError('Completed refinement fold differs.')
        model = ResNet34CV().to(device)
        checkpoint = load_refine_checkpoint(folder/f'best_{selected}.pt',signature)
        model.load_state_dict(checkpoint['model'],strict=True)
        return model,done,valid_loader
    seed_everything(config['seed'])
    torch.set_num_threads(config['cpu_threads'])
    train_set = RefinementDataset(development,cache,config,fold,'train')
    loader = DataLoader(train_set,batch_size=config['batch_size'],shuffle=True,num_workers=0)
    last = folder/'last.pt'
    model = ResNet34CV(None if last.exists() else weights).to(device)
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    optimizer = torch.optim.AdamW([
        dict(params=model.backbone.parameters(),lr=config['lr_backbone']),
        dict(params=[p for p in model.parameters() if id(p) not in backbone_ids],lr=config['lr_decoder'])],weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
        lr_lambda=lambda step:.05+.95*(1+math.cos(math.pi*step/config['epochs']))/2)
    scaler = torch.amp.GradScaler('cuda')
    history, start, times = [], 0, []
    if last.exists():
        state = load_refine_checkpoint(last,signature,optimizer=True)
        model.load_state_dict(state['model'],strict=True)
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history,start = state['history'],int(state['epoch'])
        random.setstate(state['python_rng'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state(state['cuda_rng'],device=0)
        del state
        print(f'Resume refinement fold {fold} after epoch {start}.',flush=True)
    tracked = ('rmse_coordinate_px','mean_distance_px')
    best = {key:dict(score=float('inf'),epoch=0) for key in tracked}
    for key in tracked:
        if (folder/f'best_{key}.pt').exists():
            item = load_refine_checkpoint(folder/f'best_{key}.pt',signature)
            best[key] = dict(score=float(item['metrics'][key]),epoch=int(item['epoch']))
            del item
    torch.cuda.reset_peak_memory_stats(device=0)
    for epoch in range(start+1,config['epochs']+1):
        begin = time.perf_counter()
        model.train()
        total,count,skipped = 0.,0,0
        for images,points,_ in loader:
            optimizer.zero_grad(set_to_none=True)
            images,points = images.to(device),points.to(device)
            with torch.autocast(device_type='cuda',enabled=True):
                logits = model(images)
                if logits.shape != (len(images),96,96):
                    raise RuntimeError('Expected a 96x96 refinement heatmap.')
                loss = heatmap_loss(logits,points,sigma=config['sigma'])
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite refinement loss.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            previous = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < previous)
            total += float(loss.detach())*len(images)
            count += len(images)
        if skipped == len(loader) and history and history[-1]['amp_skipped_steps'] == len(loader):
            raise RuntimeError('Every optimizer step skipped for two epochs.')
        validation = predict_refinement(model,valid_loader,device,config,coarse['validation'][fold],fold)
        metrics = metrics_from_rows(validation)
        for key in tracked:
            if metrics[key] < best[key]['score']:
                best[key] = dict(score=metrics[key],epoch=epoch)
                atomic_torch_save(folder/f'best_{key}.pt',dict(model=model.state_dict(),epoch=epoch,
                    signature=signature,metrics=metrics,train_config=config))
        applied_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        row = dict(epoch=epoch,train_heatmap_ce=total/count,lr_backbone=applied_lr,**metrics,
            validation_fallbacks=sum(int(r['fallback']) for r in validation),amp_skipped_steps=skipped,
            gpu_peak_mib=torch.cuda.max_memory_allocated(device=0)/1024**2,
            compute_seconds=time.perf_counter()-begin)
        history.append(row)
        if epoch % config['checkpoint_every'] == 0 or epoch == config['epochs']:
            atomic_torch_save(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),
                scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch,history=history,
                signature=signature,python_rng=random.getstate(),numpy_rng=np.random.get_state(),
                torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state(device=0)))
        times.append(time.perf_counter()-begin)
        atomic_csv(folder/'history.csv',history)
        save_json(folder/'status.json',dict(state='training',epoch=epoch,best=best,updated_unix=time.time()))
        refinement_eta(output,epoch,times,config)
        print(f'Refine fold{fold} {epoch:03}/{config["epochs"]} CE={total/count:.4f}'
              f' RMSE={metrics["rmse_coordinate_px"]:.2f}px MRE={metrics["mean_distance_px"]:.2f}px'
              f' best={best[selected]["score"]:.2f}px fallback={row["validation_fallbacks"]}'
              f' AMPskip={skipped} {times[-1]:.1f}s',flush=True)
    done = dict(signature=signature,best=best,selection_metric=selected,epochs=config['epochs'])
    save_json(complete_file,done)
    model.load_state_dict(load_refine_checkpoint(folder/f'best_{selected}.pt',signature)['model'],strict=True)
    return model,done,valid_loader


def refine_fold_config(protocol,fold):
    return dict(protocol,fold=fold,seed=protocol['split_seed']+100+fold)


def refinement_reports(data, train, test, coarse, output, protocol, render=True):
    records, tests = train.to_dict('records'),test.to_dict('records')
    val_folds = {f:[] for f in range(5)}
    test_folds, fold_metrics = {},[]
    available = []
    for fold in range(5):
        folder = output/f'fold_{fold}'
        if not (folder/'fold_complete.json').exists():
            continue
        expected = stable_hash(refine_fold_config(protocol,fold))
        done = json.loads((folder/'fold_complete.json').read_text())
        if done['signature'] != expected or not done['predictions_complete']:
            raise ValueError('Refinement fold completion metadata differs.')
        rows = read_rows(folder/'validation_refine.csv')
        held = [r for r in records if int(r['fold']) == fold]
        val_folds[fold] = validate_oof(rows,held)
        validate_test_rows(rows,held,fold)
        test_folds[fold] = read_rows(folder/'test_refine.csv')
        validate_test_rows(test_folds[fold],tests,fold)
        available.append(fold)
        fold_metrics.append(dict(fold=fold,best_epoch=done['training']['best'][protocol['selection_metric']]['epoch'],
                                 **metrics_from_rows(rows)))
    if fold_metrics:
        atomic_csv(output/'fold_metrics.csv',fold_metrics)
    complete = set(available) == set(range(5))
    selection = dict(complete=complete,completed_folds=available,metric=protocol['selection_metric'],
        official_metric_confirmed=False,submitted_to_competition=False,
        coarse_source='Cached original 512 per-fold predictions; second stage initialized from ImageNet.',
        limitation='Both stages selected epochs using the same development folds; OOF is not an independent test.')
    if not complete:
        save_json(output/'selection.json',selection)
        return False
    refined = validate_oof([r for f in range(5) for r in val_folds[f]],records)
    coarse_oof = validate_oof([r for f in range(5) for r in coarse['validation'][f].values()],records)
    arms = dict(coarse512=coarse_oof,refine384=refined,
                coarse_refine_equal=blend_prediction_rows(coarse_oof,refined,True))
    arm_tests = dict(coarse512={f:list(coarse['test'][f].values()) for f in range(5)},refine384=test_folds,
        coarse_refine_equal={f:blend_prediction_rows(list(coarse['test'][f].values()),test_folds[f],False) for f in range(5)})
    comparison = []
    for arm in REFINE_ARMS:
        rows = validate_oof(arms[arm],records)
        atomic_csv(output/f'oof_{arm}.csv',rows)
        averages = average_test_folds(arm_tests[arm],tests)
        atomic_csv(output/f'test_{arm}_5fold.csv',averages)
        write_submission(data/'sample_submission.csv',averages,output/f'submission_{arm}_5fold.csv')
        comparison.append(dict(arm=arm,**metrics_from_rows(rows)))
        if render:
            worst = sorted(rows,key=lambda r:float(r['distance_px']),reverse=True)[:20]
            ordered = train.set_index('image_id').loc[[int(r['image_id']) for r in worst]].reset_index()
            points = [[float(r['Fovea_X'])/int(r['width']),float(r['Fovea_Y'])/int(r['height'])] for r in worst]
            contact_sheet(data,ordered,output/f'worst_examples_{arm}.png',points)
    comparison.sort(key=lambda r:r[protocol['selection_metric']])
    atomic_csv(output/'comparison.csv',comparison)
    selection.update(selected_arm=comparison[0]['arm'],
        validation_fallbacks=sum(int(r['fallback']) for r in refined),
        test_model_fallbacks=sum(int(r['fallback']) for f in range(5) for r in test_folds[f]),
        public_reference_score=12.31624,public_reference_file='submission_resnet34_fulltrain.csv')
    shutil.copy2(output/f'submission_{selection["selected_arm"]}_5fold.csv',output/'submission_selected_refine_5fold.csv')
    save_json(output/'selection.json',selection)
    print('\nFive-fold development comparison:',flush=True)
    print(pd.DataFrame(comparison).to_string(index=False),flush=True)
    return True


def refine_light_archive(output):
    target = output/'task3_refine_results_light.zip'
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in output.rglob('*'):
            if path.is_file() and path.suffix in {'.csv','.json','.png','.md'}:
                archive.write(path,path.relative_to(output))
    temporary.replace(target)


def prepare_refinement(data_dir=None,coarse_source=None,output_dir='/kaggle/working/task3_refine_cv',
                       resume_dir=None,epochs=60,batch_size=8,checkpoint_every=5,cpu_threads=2,
                       selection_metric='rmse_coordinate_px'):
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Run this Notebook on Kaggle. No local training was started.')
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU before starting; CPU training is disabled.')
    cfg = dict(epochs=epochs,batch_size=batch_size,checkpoint_every=checkpoint_every,cpu_threads=cpu_threads,
        selection_metric=selection_metric,image_size=384,crop_side=768,center_jitter=160.,max_shift=256.,
        sigma=1.5,decode_radius=3,lr_backbone=5e-5,lr_decoder=5e-4)
    validate_refine_config(cfg)
    data,source,output = find_data(data_dir),find_coarse_source(coarse_source),Path(output_dir).resolve()
    if source.is_dir() and (output == source or output.is_relative_to(source) or source.is_relative_to(output)):
        raise ValueError('Keep refinement and coarse output directories separate.')
    manifest = load_manifest(data)
    # Read the previous split seed; matching is then checked against all current pixels/labels.
    bundle = CoarseBundle(source)
    try:
        split_seed = int(bundle.json('protocol.json')['seed'])
    finally:
        bundle.close()
    train = assign_folds(manifest[manifest.subset=='train'],split_seed,5)
    test = manifest[manifest.subset=='test'].copy()
    if len(train) != 80 or len(test) != 20:
        raise ValueError('Unexpected image counts.')
    coarse = validate_coarse_bundle(source,train.to_dict('records'),test.to_dict('records'))
    protocol = dict(version=REFINE_VERSION,split_seed=split_seed,folds=5,**cfg,
        source_sha256=refinement_sources(),coarse_fingerprint=coarse['fingerprint'],
        initialization='Official ImageNet ResNet34; no coarse-model weight warm start',
        train_crop='Ground-truth center plus independent uniform +/-160 native pixel jitter per axis',
        inference_crop='Matching held-out/test coarse512 prediction; zero GT geometry access',
        fallback='Coarse point if refined point outside image or Euclidean shift >256 native pixels',
        augmentation='hflip0.5 affine8deg scale0.95-1.05 translate0.02 colorjitter; no flip TTA')
    output.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(output).free < 3*1024**3:
        raise RuntimeError('Need at least 3 GiB free for models, optimizer and atomic checkpoint writes.')
    restore_backup(resume_dir,output,protocol)
    if (output/'protocol.json').exists() and json.loads((output/'protocol.json').read_text()) != protocol:
        raise ValueError('Existing refinement protocol differs. Use a new OUTPUT_DIR.')
    save_json(output/'protocol.json',protocol)
    save_json(output/'coarse_source.json',dict(path=str(source),fingerprint=coarse['fingerprint'],
        protocol=coarse['protocol'],baseline_metrics=coarse['baseline_metrics']))
    train.to_csv(output/'split.csv',index=False)
    manifest.to_csv(output/'manifest.csv',index=False)
    coverage = []
    for f in range(5):
        for row in coarse['validation'][f].values():
            box = crop_geometry((float(row['Fovea_X']),float(row['Fovea_Y'])),cfg['crop_side'])
            x,y = point_to_patch((float(row['gt_x']),float(row['gt_y'])),box)
            coverage.append(dict(image_id=int(row['image_id']),fold=f,gt_inside_crop=int(0<=x<1 and 0<=y<1)))
    atomic_csv(output/'coarse_crop_coverage.csv',coverage)
    save_json(output/'environment.json',dict(torch=torch.__version__,numpy=np.__version__,
        gpu=torch.cuda.get_device_name(0),visible_gpu_count=torch.cuda.device_count(),
        device_policy='cuda:0 sequential folds; workers=0; original RGB cache up to about 2 GiB'))
    print('Previous 512 coarse OOF:',coarse['baseline_metrics'],flush=True)
    print('Held-out GT inside fixed crops:',sum(r['gt_inside_crop'] for r in coverage),'/80; diagnostic only, not used to move crops.',flush=True)
    return dict(data=data,source=source,output=output,manifest=manifest,train=train,test=test,coarse=coarse,protocol=protocol)


def run_refinement(prepared,run_folds=(0,1,2,3,4),weights=None,keep_last=False):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU; local/CPU training is disabled.')
    if not run_folds or len(set(run_folds)) != len(run_folds) or not set(run_folds) <= set(range(5)):
        raise ValueError('Invalid fold schedule.')
    data,output,train,test,coarse,protocol = [prepared[k] for k in ('data','output','train','test','coarse','protocol')]
    pending = [f for f in run_folds if not (output/f'fold_{f}/fold_complete.json').exists()]
    if pending:
        cached = prepared['source']/'_pretrained/resnet34-b627a593.pth'
        weights = resolve_weights(output,weights or (cached if cached.is_file() else None))
    cache = OriginalImageCache(data)
    with training_lock(output):
        try:
            for fold in run_folds:
                config = refine_fold_config(protocol,fold)
                signature = stable_hash(config)
                folder = output/f'fold_{fold}'
                if (folder/'fold_complete.json').exists():
                    done = json.loads((folder/'fold_complete.json').read_text())
                    if done['signature'] != signature or not done['predictions_complete']:
                        raise ValueError('Completed refinement protocol differs.')
                    if any(not (folder/f'best_{key}.pt').is_file() for key in ('rmse_coordinate_px','mean_distance_px')):
                        raise ValueError('Completed fold lacks weights. Mount the FULL refinement output, not its light ZIP.')
                    print(f'Reuse completed refinement fold {fold}.',flush=True)
                    continue
                development = train[train.fold!=fold].to_dict('records')
                held = train[train.fold==fold].to_dict('records')
                save_json(output/'status.json',dict(state='running',fold=fold,updated_unix=time.time()))
                model,done,valid_loader = train_refine_fold(development,held,cache,coarse,folder,config,weights,output)
                validation = predict_refinement(model,valid_loader,torch.device('cuda:0'),config,coarse['validation'][fold],fold)
                test_loader = DataLoader(RefinementDataset(test.to_dict('records'),cache,config,fold,'test',coarse['test'][fold]),
                    batch_size=config['batch_size'],shuffle=False,num_workers=0)
                predictions = predict_refinement(model,test_loader,torch.device('cuda:0'),config,coarse['test'][fold],fold)
                atomic_csv(folder/'validation_refine.csv',validation)
                atomic_csv(folder/'test_refine.csv',predictions)
                save_json(folder/'fold_complete.json',dict(signature=signature,training=done,
                    predictions_complete=True,updated_unix=time.time()))
                save_json(folder/'status.json',dict(state='completed',best=done['best'],updated_unix=time.time()))
                if not keep_last and (folder/'last.pt').exists():
                    (folder/'last.pt').unlink()
                del model,valid_loader,test_loader
                gc.collect()
                torch.cuda.empty_cache()
                refinement_reports(data,train,test,coarse,output,protocol,render=False)
                refine_light_archive(output)
            complete = refinement_reports(data,train,test,coarse,output,protocol)
            save_json(output/'status.json',dict(state='completed' if complete else 'partial_completed',updated_unix=time.time()))
            refine_light_archive(output)
        except BaseException as error:
            save_json(output/'status.json',dict(state='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',
                error=str(error),updated_unix=time.time()))
            refine_light_archive(output)
            raise
    return output


## 2. 预检查与验证边界

先核对所有80张图的像素哈希、精确坐标、折号和逐折粗预测，确保0039与0077仍同折。每张验证图只使用**对应留出折**的512粗预测，测试图也使用对应折的粗预测，再平均最终五折结果。

训练裁剪中心是本折训练标签加每轴±160原图像素的均匀随机偏移，以模拟粗定位误差。验证/测试裁剪几何只读取粗预测，真实坐标仅在预测后用于评分。预检查显示标签是否落入裁剪区，这只是诊断，不会据此移动裁剪框。

超出图像的裁剪区域补黑，保留真实原图到裁剪的坐标偏移。精修点若落到原图之外，或与粗定位相距超过预先固定的256像素，则回退粗点；次数与原始精修坐标会记录。这个阈值不根据本次OOF或公开分数搜索。


In [ ]:
PREPARED = prepare_refinement(
    data_dir=DATA_DIR, coarse_source=COARSE_SOURCE, output_dir=OUTPUT_DIR,
    resume_dir=RESUME_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY, cpu_threads=CPU_THREADS,
    selection_metric=SELECTION_METRIC,
)
print('GPU:', torch.cuda.get_device_name(0))
print('Coarse source:', PREPARED['source'])
print('New outputs:', PREPARED['output'])
display(pd.read_csv(OUTPUT_DIR/'coarse_crop_coverage.csv').groupby('fold').agg(
    images=('image_id','size'), inside_fixed_crop=('gt_inside_crop','sum')))


## 3. 新训练5个精修模型

模型结构与此前ResNet34热图头一致，固定编码器BatchNorm运行统计；输入384、输出96×96热图，σ=1.5、局部解码半径3。AdamW编码器/头学习率5e-5/5e-4，weight_decay=1e-4，60轮余弦衰减至各自初始值5%，AMP与梯度裁剪。训练增强为小幅仿射、颜色变化、水平翻转；**无翻转TTA**。

原图RGB按需缓存，完整100张约占2 GiB内存；workers=0。每3个实际epoch后开始估算时间，预计时间以当前会话日志为准。逐折保存RMSE和MRE两套最佳权重，每5轮保存优化器断点。最佳文件可能比最近优化器断点新，恢复时会核对配置并保留已评估的最佳模型。

需要分批时仅修改RUN_FOLDS；修改轮数、参数或实现必须使用新输出目录。官方评分公式尚未确认，默认按坐标RMSE选主权重。


In [ ]:
RESULT_DIR = run_refinement(PREPARED, run_folds=RUN_FOLDS, weights=WEIGHTS,
    keep_last=KEEP_LAST_CHECKPOINT)


## 4. 完整五折结果与候选

比较三种固定方案：原512粗定位、局部精修（含固定回退）、粗与精修等权平均。不拟合融合权重，不按图像挑选模型。只在五折齐全时导出候选与自动选定文件；精修无收益时允许自动选回原粗定位。

选轮次和方案都使用验证标签，且粗定位轮次也曾参考相同留出折，因此属于开发评估，不是完全独立的泛化分数。五折测试结果每个模型只训练约64张，不可直接将其OOF与原全量版公开12.31624判优劣。此Notebook不提交比赛。


In [ ]:
from IPython.display import display, Image as DisplayImage
selection = json.loads((RESULT_DIR/'selection.json').read_text())
print(json.dumps(selection,ensure_ascii=False,indent=2))
if (RESULT_DIR/'comparison.csv').exists():
    display(pd.read_csv(RESULT_DIR/'comparison.csv'))
    display(pd.read_csv(RESULT_DIR/'fold_metrics.csv'))
    for arm in REFINE_ARMS:
        path = RESULT_DIR/f'worst_examples_{arm}.png'
        if path.exists():
            print(arm, '：青色标签、黄色预测')
            display(DisplayImage(filename=str(path)))
    print('Candidate:', RESULT_DIR/'submission_selected_refine_5fold.csv')
print('Light ZIP:', RESULT_DIR/'task3_refine_results_light.zip')


## 5. 完整备份（默认执行，无需重训）

轻量ZIP只含分析文件。下面额外打包本版完整目录，包含10份最佳权重（5折×2指标）、配置、预测与日志；ZIP在结果目录外，避免把自身再次打包。下载右侧Output中的 `task3_refine_full_backup.zip`；体积较大，建议本地保存到D盘。

如后续在新会话恢复本版实验：把**本版完整Output解压目录**挂载为Input，设置RESUME_DIR为直接包含本版protocol.json的目录；添加比赛图片，继续使用本修正版内置的原粗预测缓存。仅轻量ZIP无法恢复训练。完整折默认已删除last.pt，保留最优模型与预测，可跳过已完成折；未完成折按最近5轮断点续跑。


In [ ]:
if CREATE_FULL_BACKUP and selection.get('complete'):
    expected = [RESULT_DIR/f'fold_{fold}/best_{metric}.pt' for fold in range(5)
                for metric in ('rmse_coordinate_px','mean_distance_px')]
    if not all(path.is_file() for path in expected):
        raise RuntimeError('缺少精修最佳权重，不能生成完整备份。')
    full_zip = RESULT_DIR.parent/'task3_refine_full_backup.zip'
    with zipfile.ZipFile(full_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for path in sorted(RESULT_DIR.rglob('*')):
            if path.is_file():
                archive.write(path,path.relative_to(RESULT_DIR))
    with zipfile.ZipFile(full_zip) as archive:
        assert all(path.relative_to(RESULT_DIR).as_posix() in archive.namelist() for path in expected)
    print('Full backup ready:',full_zip, f'{full_zip.stat().st_size/1024**3:.2f} GiB; 10 best weights included.')
else:
    print('本次未生成完整备份：请完成全部五折并将CREATE_FULL_BACKUP设为True。')


本地已检查的内容见配套检查报告；准备阶段未运行本轮GPU训练。GPU前向、反向、真实优化器恢复和最终误差图会在Kaggle执行。首次批次会检查96×96输出、有限损失与AMP跳步。保留12.31624原全量版，不覆盖已有CSV或模型。
